# z013 - Variables manuales (FE intra mes, punto 1.3 del profesor)

**Idea:** los árboles cortan de a una variable; les cuesta ver relaciones entre dos (por ejemplo "usa el 90% del límite").
Se las damos calculadas. Se crean **antes** de los lags, así también tienen lag y delta.

Trial 34, 538 árboles, 10 semillas, train 202104, valid 202106. Versión "Actual" = sin las variables nuevas
(entrenada con las mismas columnas que siempre, sacando las nuevas y sus lags).

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331,   # las tuyas
             111119, 222227, 333331, 444443, 555557]  # 5 más (las mismas del ensamble)
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2a. Cargar datos ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)
df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]

In [3]:
# --- 2b. Variables manuales ---
def div(a, b):
    # División que devuelve NaN si el denominador es 0
    return (a / b.replace(0, np.nan)).astype("float32")

nuevas_vars = {
    # Tarjetas: totales Visa + Master
    "tc_consumo_total":   df["Visa_mconsumototal"].fillna(0) + df["Master_mconsumototal"].fillna(0),
    "tc_saldo_total":     df["Visa_msaldototal"].fillna(0) + df["Master_msaldototal"].fillna(0),
    "tc_limite_total":    df["Visa_mlimitecompra"].fillna(0) + df["Master_mlimitecompra"].fillna(0),
    "tc_pagado_total":    df["Visa_mpagado"].fillna(0) + df["Master_mpagado"].fillna(0),
    "tc_cconsumos_total": df["Visa_cconsumos"].fillna(0) + df["Master_cconsumos"].fillna(0),
}
nuevas_vars["tc_uso_limite"]      = div(nuevas_vars["tc_saldo_total"], nuevas_vars["tc_limite_total"])
nuevas_vars["tc_consumo_limite"]  = div(nuevas_vars["tc_consumo_total"], nuevas_vars["tc_limite_total"])
nuevas_vars["tc_pagado_saldo"]    = div(nuevas_vars["tc_pagado_total"], nuevas_vars["tc_saldo_total"])
nuevas_vars["visa_uso_limite"]    = div(df["Visa_msaldototal"], df["Visa_mlimitecompra"])
nuevas_vars["master_uso_limite"]  = div(df["Master_msaldototal"], df["Master_mlimitecompra"])
nuevas_vars["visa_pago_minimo"]   = div(df["Visa_mpagado"], df["Visa_mpagominimo"])
# Productos y relación con el banco
nuevas_vars["cant_productos"] = (df[["ctarjeta_visa", "ctarjeta_master", "cprestamos_personales", "cprestamos_prendarios",
                                     "cprestamos_hipotecarios", "cseguro_vida", "cseguro_auto", "cseguro_vivienda",
                                     "cseguro_accidentes_personales", "cpayroll_trx"]] > 0).sum(axis=1)
nuevas_vars["prestamos_total"] = (df["mprestamos_personales"].fillna(0) + df["mprestamos_prendarios"].fillna(0)
                                  + df["mprestamos_hipotecarios"].fillna(0))
# Plata y actividad
nuevas_vars["saldo_sobre_payroll"]     = div(df["mcuentas_saldo"], df["mpayroll"])
nuevas_vars["prestamos_sobre_payroll"] = div(nuevas_vars["prestamos_total"], df["mpayroll"])
nuevas_vars["trx_sobre_antiguedad"]    = div(df["ctrx_quarter"], df["cliente_antiguedad"])
nuevas_vars["antiguedad_sobre_edad"]   = div(df["cliente_antiguedad"], df["cliente_edad"] * 12)
nuevas_vars["rentabilidad_mensual_vs_anual"] = div(df["mrentabilidad"], df["mrentabilidad_annual"])

df = pd.concat([df, pd.DataFrame(nuevas_vars).astype("float32")], axis=1)
sin_lags = []
print(len(nuevas_vars), "variables nuevas:", list(nuevas_vars))

18 variables nuevas: ['tc_consumo_total', 'tc_saldo_total', 'tc_limite_total', 'tc_pagado_total', 'tc_cconsumos_total', 'tc_uso_limite', 'tc_consumo_limite', 'tc_pagado_saldo', 'visa_uso_limite', 'master_uso_limite', 'visa_pago_minimo', 'cant_productos', 'prestamos_total', 'saldo_sobre_payroll', 'prestamos_sobre_payroll', 'trx_sobre_antiguedad', 'antiguedad_sobre_edad', 'rentabilidad_mensual_vs_anual']


In [4]:
# --- 2c. Lags y deltas de orden 1 y 2 (sobre todas las columnas, incluidas las nuevas) ---
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente" and c not in sin_lags]
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 851


In [5]:
# --- 3. Funciones de evaluación ---
def metricas(prob, mascara=None):
    # Ganancia acumulada ordenando de mayor a menor probabilidad (opcional: solo una parte de los clientes)
    g = ganancia_cliente if mascara is None else ganancia_cliente[mascara]
    p = prob if mascara is None else prob[mascara]
    acum = np.cumsum(g[np.argsort(p)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[min(11_999, len(acum) - 1)] / 1e6

def resumen(probs_por_version):
    """probs_por_version = {version: [prob_semilla1, ..., prob_semilla5]}"""
    # a) Por semilla: ganancia máxima, mejor N y top 12.000
    filas = []
    for v, lista in probs_por_version.items():
        for s, pr in zip(SEMILLAS, lista):
            g, n, g12 = metricas(pr)
            filas.append({"version": v, "semilla": s, "ganancia_max_M": round(g, 1), "mejor_n": n, "top12000_M": round(g12, 1)})
    por_semilla = pd.DataFrame(filas)
    tabla = por_semilla.groupby("version", sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

    # b) Shuffle Split del test: ensamble de las 5 semillas evaluado en 50 particiones del 50% de 202106
    rng = np.random.default_rng(SEMILLAS[0])
    ens = {v: np.mean(lista, axis=0) for v, lista in probs_por_version.items()}
    part = []
    for i in range(N_PARTICIONES):
        mascara = rng.random(len(ganancia_cliente)) < 0.5
        part.append({v: metricas(pr, mascara)[0] for v, pr in ens.items()})
    part = pd.DataFrame(part)
    tabla[("shuffle_split", "mean")] = part.mean().round(1)
    tabla[("shuffle_split", "std")] = part.std().round(1)
    tabla[("shuffle_split", "veces_mejor")] = part.idxmax(axis=1).value_counts().reindex(tabla.index).fillna(0).astype(int)
    tabla[("ensamble_5", "ganancia_max_M")] = [round(metricas(ens[v])[0], 1) for v in tabla.index]
    return por_semilla, tabla

def anotar(tabla, prefijo, meses_train, extra=""):
    ruta_res = "resultados_experimentos.csv"
    res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
    res = res[~res["descripcion"].str.startswith(prefijo)]   # sin duplicar si se vuelve a correr
    for v, r in tabla.iterrows():
        nueva = {
            "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
            "descripcion": (f"{prefijo} {v}: prom. {len(SEMILLAS)} semillas; shuffle split {r[('shuffle_split', 'mean')]} "
                            f"(mejor en {r[('shuffle_split', 'veces_mejor')]}/{N_PARTICIONES}){extra}"),
            "meses_train": str(meses_train), "mes_valid": MES_VALID, "semilla": str(SEMILLAS),
            "params": str(params), "mejor_n": int(r[("mejor_n", "mean")]),
            "ganancia_max_M": r[("ganancia_max_M", "mean")], "ganancia_top12000_M": r[("top12000_M", "mean")],
        }
        res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)
    res["n_enviados"] = res["n_enviados"].astype("Int64")
    res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
    print("Anotado en", ruta_res)

In [6]:
# --- 4. Entrenar las dos versiones ---
nombres_nuevos = list(nuevas_vars)
vars_actual = [v for v in variables if not any(v == c or v.startswith(c + "_") for c in nombres_nuevos)]
train = df[df["foto_mes"] == 202104]
probs = {}
for version, vars_v in [("Actual", vars_actual), ("Con manuales", variables)]:
    datos = lgb.Dataset(train[vars_v], label=train["clase_binaria"])
    modelos = [lgb.train({**params, "seed": s}, datos, num_boost_round=ARBOLES) for s in SEMILLAS]
    probs[version] = [m.predict(valid[vars_v]) for m in modelos]
    print(version, len(vars_v), "variables, listo")

# Importancia de las variables nuevas (promedio de las 5 semillas, versión "Con manuales")
imp = pd.Series(np.mean([m.feature_importance("gain") for m in modelos], axis=0), index=variables)
imp = (imp / imp.sum() * 100).sort_values(ascending=False)
print("Importancia total de las variables nuevas (con lags/deltas):",
      round(imp[[v for v in variables if v not in vars_actual]].sum(), 1), "%")
print(imp[[v for v in imp.index if v not in vars_actual]].head(15).round(2))

Actual 761 variables, listo


Con manuales 851 variables, listo
Importancia total de las variables nuevas (con lags/deltas): 23.3 %
cant_productos                        3.28
trx_sobre_antiguedad                  2.28
prestamos_total                       1.55
trx_sobre_antiguedad_lag1             1.43
saldo_sobre_payroll                   1.10
cant_productos_lag1                   1.04
tc_saldo_total                        1.02
visa_uso_limite                       0.92
prestamos_total_lag1                  0.83
prestamos_sobre_payroll_delta1        0.76
saldo_sobre_payroll_delta1            0.74
prestamos_sobre_payroll               0.74
rentabilidad_mensual_vs_anual         0.69
rentabilidad_mensual_vs_anual_lag1    0.67
tc_uso_limite                         0.47
dtype: float64


In [7]:
# --- 5. Comparación ---
por_semilla, tabla = resumen(probs)
print(por_semilla.pivot(index="semilla", columns="version", values="ganancia_max_M"))
tabla

version  Actual  Con manuales
semilla                      
111119    396.8         392.2
222227    396.8         399.6
333331    403.6         399.3
444443    396.9         397.5
501419    400.5         392.8
555557    401.0         395.8
623353    397.2         400.4
786349    401.4         395.5
834569    397.2         391.9
946331    401.8         401.0


ganancia_max_M      top12000_M       mejor_n          \
                       mean  std       mean  std     mean     std   
version                                                             
Actual                399.3  2.6      391.7  3.7  13230.9  1224.2   
Con manuales          396.6  3.5      384.4  4.8  14302.6   617.1   

             shuffle_split                       ensamble_5  
                      mean   std veces_mejor ganancia_max_M  
version                                                      
Actual               198.9  11.8          40          400.5  
Con manuales         196.4  12.5          10          396.6

In [8]:
# --- 6. Anotar ---
anotar(tabla, "z013 variables manuales", [202104])

Anotado en resultados_experimentos.csv


## Conclusiones (08/10)
- Con manuales: **396,6 M** vs 399,3 → **−2,7 M** (Shuffle Split: Actual mejor 40/50). No suman. Descartado.